# 07 — Preparação dos dados (ADHD-200)

Objetivo: transformar as matrizes de conectividade funcional em atributos para os modelos de classificação.

Etapas:

1. carregar e validar a tabela de participantes e as matrizes FC;
2. aplicar a transformação de Fisher z às correlações, com proteção contra valores exatamente iguais a -1 ou 1;
3. vetorizar o triângulo superior sem a diagonal, gerando 17.955 atributos por participante;
4. criar folds estratificados agrupados por centro de aquisição (site), sem separar participantes do mesmo centro entre treino e teste;
5. salvar o conjunto preparado em data/processed/dataset.npz.

> Decisões metodológicas: a amostra permanece com 709 participantes (453 controles e 256 participantes com TDAH). A transformação de Fisher z é aplicada elemento a elemento, sem estimar parâmetros a partir da amostra. Qualquer padronização, seleção de atributos ou ajuste de hiperparâmetros deve ser aprendido apenas com os dados de treino de cada fold nos notebooks de modelagem.

### 1. Localizar a raiz do projeto e importar a configuração

In [13]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# 1. Localizar a raiz do projeto e importar config.py
diretorio_atual = Path.cwd().resolve()
candidatos = [diretorio_atual, *diretorio_atual.parents]
raiz_projeto = next(
    (p for p in candidatos if (p / 'config.py').is_file() and (p / 'notebooks').is_dir()),
    None,
)
if raiz_projeto is None:
    raise FileNotFoundError(
        f'Não encontrei a raiz do projeto a partir de {diretorio_atual}. '
        'Abra este notebook dentro de TCC_TDAH/notebooks.'
    )
if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))
import config

config.criar_pastas()
print('Raiz do projeto:', raiz_projeto)
print('Arquivo de participantes:', config.ARQ_PARTICIPANTES)
print('Pasta de conectividade:', config.CONNECTIVITY)
print('Arquivo de saída:', config.ARQ_DATASET)

# 2. Carregar a tabela e adaptar os nomes reais das colunas.
if not config.ARQ_PARTICIPANTES.is_file():
    raise FileNotFoundError(f'Arquivo não encontrado: {config.ARQ_PARTICIPANTES}')
if not config.CONNECTIVITY.is_dir():
    raise FileNotFoundError(f'Pasta não encontrada: {config.CONNECTIVITY}')

df_original = pd.read_csv(config.ARQ_PARTICIPANTES, dtype=str)
aliases = {
    config.ID_COLUMN: [config.ID_COLUMN, 'ID_padronizado', 'subject_id', 'participant_id', 'ID'],
    config.TARGET_COLUMN: [config.TARGET_COLUMN, 'classe', 'class', 'label', 'target'],
    config.SITE_COLUMN: [config.SITE_COLUMN, 'Site', 'SITE'],
}
renomear = {}
for nome_padrao, opcoes in aliases.items():
    encontrada = next((c for c in opcoes if c in df_original.columns), None)
    if encontrada is None:
        raise ValueError(
            f'Não encontrei uma coluna equivalente a {nome_padrao!r}. '
            f'Colunas disponíveis: {df_original.columns.tolist()}'
        )
    if encontrada != nome_padrao:
        renomear[encontrada] = nome_padrao

df = df_original.rename(columns=renomear).copy()
df[config.ID_COLUMN] = df[config.ID_COLUMN].astype('string').str.strip()
df[config.SITE_COLUMN] = df[config.SITE_COLUMN].astype('string').str.strip()
df[config.TARGET_COLUMN] = pd.to_numeric(df[config.TARGET_COLUMN], errors='coerce')

if df[[config.ID_COLUMN, config.SITE_COLUMN, config.TARGET_COLUMN]].isna().any().any():
    raise ValueError('Há IDs, sites ou classes ausentes na tabela de participantes.')
if df[config.ID_COLUMN].duplicated().any():
    raise ValueError('Há IDs duplicados na tabela de participantes.')
df[config.TARGET_COLUMN] = df[config.TARGET_COLUMN].astype(int)
classes_encontradas = set(df[config.TARGET_COLUMN].unique())
classes_esperadas = {int(config.CONTROL_LABEL), int(config.ADHD_LABEL)}
if classes_encontradas != classes_esperadas:
    raise ValueError(f'Classes encontradas: {classes_encontradas}; esperadas: {classes_esperadas}.')

ids = df[config.ID_COLUMN].astype(str).to_numpy()
y = df[config.TARGET_COLUMN].to_numpy(dtype=np.int64)
sites = df[config.SITE_COLUMN].astype(str).to_numpy()

print('\nParticipantes:', len(df))
print('Distribuição das classes:')
print(pd.Series(y).value_counts().sort_index().rename(index={0: 'Controle (0)', 1: 'TDAH (1)'}))
print('\nParticipantes por site:')
print(pd.crosstab(pd.Series(sites, name='site'), pd.Series(y, name='classe')))


Raiz do projeto: /home/maria/TCC_TDAH
Arquivo de participantes: /home/maria/TCC_TDAH/data/processed/participants.csv
Pasta de conectividade: /home/maria/TCC_TDAH/data/connectivity
Arquivo de saída: /home/maria/TCC_TDAH/data/processed/dataset.npz

Participantes: 709
Distribuição das classes:
Controle (0)    453
TDAH (1)        256
Name: count, dtype: int64

Participantes por site:
classe       0    1
site               
KKI         58   22
NYU         87  103
NeuroIMAGE  21   20
OHSU        38   34
Peking_1    59   23
Peking_2    32   35
Peking_3    23   19
Pittsburgh  88    0
WashU       47    0


### 2. Validação das matrizes

In [14]:
# 3. Carregar e validar as matrizes de conectividade
caminhos_fc = [config.CONNECTIVITY / f'FC_{pid}.npy' for pid in ids]
faltantes = [str(p) for p in caminhos_fc if not p.is_file()]
if faltantes:
    raise FileNotFoundError(f'Faltam {len(faltantes)} matrizes. Exemplos: {faltantes[:5]}')

fc = np.stack([np.load(p, allow_pickle=False) for p in caminhos_fc]).astype(np.float64, copy=False)
forma_esperada = (len(df), int(config.N_ROIS), int(config.N_ROIS))
if fc.shape != forma_esperada:
    raise ValueError(f'Formato das matrizes: {fc.shape}; esperado: {forma_esperada}.')
if not np.isfinite(fc).all():
    raise ValueError('As matrizes FC contêm NaN ou infinito.')
if np.any(fc < -1.000001) or np.any(fc > 1.000001):
    raise ValueError('Há valores fora do intervalo de correlação [-1, 1].')
if not np.allclose(fc, np.swapaxes(fc, 1, 2), atol=1e-6):
    raise ValueError('Uma ou mais matrizes FC não são simétricas.')

print('Formato das matrizes FC:', fc.shape)
print('Tipo numérico de entrada:', fc.dtype)
print('Todas as matrizes são finitas e simétricas.')


Formato das matrizes FC: (709, 190, 190)
Tipo numérico de entrada: float64
Todas as matrizes são finitas e simétricas.


In [15]:
# 4. Fisher z e vetorização do triângulo superior
epsilon = 1e-7
fc_limitada = np.clip(fc, -1.0 + epsilon, 1.0 - epsilon)
fc_fisher = np.arctanh(fc_limitada)

roi_i, roi_j = np.triu_indices(int(config.N_ROIS), k=1)
X = fc_fisher[:, roi_i, roi_j].astype(np.float32)

n_atributos_esperado = int(config.N_ROIS) * (int(config.N_ROIS) - 1) // 2
if X.shape != (len(df), n_atributos_esperado):
    raise ValueError(f'Formato de X: {X.shape}; esperado {(len(df), n_atributos_esperado)}.')
if not np.isfinite(X).all():
    raise ValueError('X contém NaN ou infinito após Fisher z.')

print('Formato de X:', X.shape)
print('Atributos por participante:', X.shape[1])
print('Valores finitos em X:', bool(np.isfinite(X).all()))
print('Intervalo observado após Fisher z:', float(X.min()), 'a', float(X.max()))


Formato de X: (709, 17955)
Atributos por participante: 17955
Valores finitos em X: True
Intervalo observado após Fisher z: -1.5912327766418457 a 1.7097053527832031


In [16]:
# 5. Criar folds estratificados agrupados por site
# Cada site fica inteiramente no treino ou no teste de um fold.
n_folds = int(config.N_FOLDS)
cv = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=int(config.SEED))
fold_id = np.full(len(df), -1, dtype=np.int8)
resumo_folds = []

for numero_fold, (idx_treino, idx_teste) in enumerate(cv.split(X, y, groups=sites)):
    if np.any(fold_id[idx_teste] != -1):
        raise RuntimeError('Algum participante foi atribuído a mais de um fold de teste.')
    fold_id[idx_teste] = numero_fold

    classes_teste = set(np.unique(y[idx_teste]))
    if classes_teste != classes_esperadas:
        sites_teste = sorted(set(sites[idx_teste]))
        raise ValueError(
            f'O fold {numero_fold} não contém as duas classes. Sites de teste: {sites_teste}; '
            f'classes encontradas: {sorted(classes_teste)}. Revise a estratégia de validação.'
        )

    sites_treino = set(sites[idx_treino])
    sites_teste = set(sites[idx_teste])
    if sites_treino.intersection(sites_teste):
        raise RuntimeError(f'Há sobreposição de sites no fold {numero_fold}.')

    resumo_folds.append({
        'fold': numero_fold,
        'n_treino': len(idx_treino),
        'n_teste': len(idx_teste),
        'controles_teste': int(np.sum(y[idx_teste] == config.CONTROL_LABEL)),
        'tdah_teste': int(np.sum(y[idx_teste] == config.ADHD_LABEL)),
        'sites_teste': ', '.join(sorted(sites_teste)),
        'sobreposicao_sites': len(sites_treino.intersection(sites_teste)),
    })

if np.any(fold_id < 0):
    raise RuntimeError('Nem todos os participantes receberam um fold de teste.')

df_folds = pd.DataFrame(resumo_folds)
display(df_folds)
print('Cada participante aparece exatamente uma vez como teste:', bool(np.all(fold_id >= 0)))
print('Total de participantes atribuídos a folds:', int(np.sum(fold_id >= 0)))


,fold,n_treino,n_teste,controles_teste,tdah_teste,sites_teste,sobreposicao_sites
0,0,590,119,85,34,"OHSU, WashU",0
1,1,587,122,81,41,"KKI, Peking_3",0
2,2,580,129,109,20,"NeuroIMAGE, Pittsburgh",0
3,3,560,149,91,58,"Peking_1, Peking_2",0
4,4,519,190,87,103,NYU,0


Cada participante aparece exatamente uma vez como teste: True
Total de participantes atribuídos a folds: 709


In [18]:
# 6. Salvar o conjunto preparado e metadados dos atributos
config.ARQ_DATASET.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    config.ARQ_DATASET,
    X=X,
    y=y,
    ids=ids,
    sites=sites,
    fold_id=fold_id,
    roi_i=roi_i.astype(np.int16),
    roi_j=roi_j.astype(np.int16),
)

print('Conjunto preparado salvo em:', config.ARQ_DATASET)
print('X:', X.shape, X.dtype)
print('y:', y.shape, y.dtype)
print('ids:', ids.shape)
print('sites:', sites.shape)
print('fold_id:', fold_id.shape)
print('Fisher z aplicado; sem padronização global ou seleção de atributos.')


Conjunto preparado salvo em: /home/maria/TCC_TDAH/data/processed/dataset.npz
X: (709, 17955) float32
y: (709,) int64
ids: (709,)
sites: (709,)
fold_id: (709,)
Fisher z aplicado; sem padronização global ou seleção de atributos.
